In [ ]:
import pandas as pd


import pandas as pd
import numpy as np

import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score


from sklearn.metrics import f1_score, recall_score, precision_score, confusion_matrix, classification_report
from sklearn.svm import SVC


all = pd.read_csv('*****.csv')

df_frame_length_25 = pd.read_pickle('*****.pkl')
i = 0
df_frame_length_25['source'] = ''
while i <len(df_frame_length_25):
    df_frame_length_25['source'][i] = df_frame_length_25['name'][i].split('-')[1]
    i = i+1

In [ ]:
df_frame_length_25['gender'] = ''
i = 0
while i <len(df_frame_length_25):
    j = 0
    while j < len(all):
        if df_frame_length_25['source'][i] ==  all['filename'][j]:
            df_frame_length_25['gender'][i] = all['Sex'][j]
        j = j+1
    i = i+1

In [ ]:
df_frame_length_25['new_column'] = df_frame_length_25['feature'].apply(lambda x: x[:20])

In [ ]:
all['newlabel'] = ''
i = 0
while i < len(all):
    if all['label'][i] == 'positive':
        all['newlabel'][i] = 1
    elif all['label'][i] == 'negative':
        all['newlabel'][i] = 0
    i = i+1

In [ ]:
df_name  = all['filename']
df_label = np.int64(all['newlabel'])

In [ ]:
from sklearn import metrics

kf = StratifiedKFold(n_splits=10, random_state=0, shuffle=True )
list40 = []
for i in range(0,20):
        list40.append(str(i)) 
scaler = StandardScaler()

acc_arr = np.empty((10, 1))

acc_arr_male = np.empty((10, 1))
acc_arr_female = np.empty((10, 1))

f1_arr = np.empty((10, 1))
precision_arr = np.empty((10, 1))
recall_arr = np.empty((10, 1))
auc_arr = np.empty((10, 1))

sen_arr = np.empty((10, 1))
spe_arr = np.empty((10, 1))

ppv_arr = np.empty((10, 1))
npv_arr = np.empty((10, 1))

recall_arr = np.empty((10, 1))
pre_arr = np.empty((10, 1))

cnf_arr= []
x = 0

for train_index, test_index in kf.split(df_name, df_label):
    name_train, name_test =  df_name[train_index] ,  df_name[test_index] 
    y_train, y_test =  df_label[train_index] ,  df_label[test_index] 
    listname_train =  name_train.values.tolist()
    listname_test =  name_test.values.tolist()
    cut_listname_train = '|'.join(listname_train)
    cut_listname_test  = '|'.join(listname_test)
    train = df_frame_length_25[df_frame_length_25["name"].str.contains(cut_listname_train)]
    test  = df_frame_length_25[df_frame_length_25["name"].str.contains(cut_listname_test)]
    
    features_train = pd.DataFrame(train['new_column'].to_list(), columns= list40)
    features_test  = pd.DataFrame(test['new_column'].to_list(), columns= list40)
    df_x_train = features_train[list40]
    df_x_test  = features_test[list40]
    df_x_train = scaler.fit_transform(df_x_train)
    df_x_test  = scaler.fit_transform(df_x_test)
    
    df_y_train = np.int64(train['label'])
    df_y_test  = np.int64(test['label'])
    model = SVC(kernel='rbf',gamma=0.1, C= 1, tol=1e-5, verbose=1,max_iter=2500)
    model.fit(df_x_train, df_y_train)
    y_pred = model.predict(df_x_test)


    male_mask = test['gender'] == 'Male'
    female_mask = test['gender'] == 'Female'
    y_pred_male = model.predict(df_x_test[male_mask])
    df_y_test_male = df_y_test[male_mask]
    y_pred_female = model.predict(df_x_test[female_mask])
    df_y_test_female = df_y_test[female_mask]
    accuracy_male = accuracy_score(df_y_test_male, y_pred_male)
    accuracy_female = accuracy_score(df_y_test_female, y_pred_female)

    
    print('Accuracy Score: {:.4f}'.format(accuracy_score(df_y_test, y_pred)))
    print('Accuracy for Males: {:.4f}'.format(accuracy_male))
    print('Accuracy for Females: {:.4f}'.format(accuracy_female))

    print('SVC f1-score  : {:.4f}'.format(f1_score(df_y_test, y_pred)))
    print('SVC precision : {:.4f}'.format(precision_score(df_y_test, y_pred)))
    print('SVC recall    : {:.4f}'.format(recall_score(df_y_test, y_pred)))
    print("\n", classification_report(df_y_test, y_pred))
    
    cnf_matrix = confusion_matrix(df_y_test, y_pred)
    acc_arr[x] = accuracy_score(df_y_test, y_pred)

    acc_arr_male[x] = accuracy_score(df_y_test_male, y_pred_male)
    acc_arr_female[x] = accuracy_score(df_y_test_female, y_pred_female)

    f1_arr[x] = f1_score(df_y_test, y_pred)
    precision_arr[x] = precision_score(df_y_test, y_pred)
    recall_arr[x] = recall_score(df_y_test, y_pred)
    auc_arr[x] = metrics.roc_auc_score(df_y_test, y_pred)
    
    tn, fp, fn, tp = confusion_matrix(df_y_test, y_pred).ravel()
    specificity = tn / (tn+fp)
    sensitivity = tp / (tp+fn)
    ppv = tp /(tp+fp)
    npv = tn /(tn+fn)
    pre = tp / (tp + fp)
    
    sen_arr[x] = sensitivity
    spe_arr[x] = specificity
    ppv_arr[x] = ppv
    npv_arr[x] = npv
    pre_arr[x] = pre
    
     

    x = x+ 1


print("%0.3f accuracy with a standard deviation of %0.2f" %
      (acc_arr.mean(), acc_arr.std()))
print("%0.3f accuracy_male with a standard deviation of %0.2f" %
      (acc_arr_male.mean(), acc_arr_male.std()))
print("%0.3f accuracy_female with a standard deviation of %0.2f" %
      (acc_arr_female.mean(), acc_arr_female.std()))
print("%0.3f precision with a standard deviation of %0.2f" %
      (precision_arr.mean(), precision_arr.std()))
print("%0.3f auc with a standard deviation of %0.2f" %
      (auc_arr.mean(), auc_arr.std()))
print("%0.3f f1 score with a standard deviation of %0.2f" %
      (f1_arr.mean(), f1_arr.std()))

print("%0.3f sensitivity with a standard deviation of %0.2f" %
      (sen_arr.mean(), sen_arr.std()))

print("%0.3f specificity with a standard deviation of %0.2f" %
      (spe_arr.mean(), spe_arr.std()))

print("%0.3f ppv with a standard deviation of %0.2f" %
      (ppv_arr.mean(), ppv_arr.std()))
print("%0.3f npv with a standard deviation of %0.2f" %
      (npv_arr.mean(), npv_arr.std()))

print("%0.3f recall with a standard deviation of %0.2f" %
      (recall_arr.mean(), recall_arr.std()))

In [ ]:
acc_arr

In [ ]:
acc_arr_male

In [ ]:
acc_arr_female